# Exploratory data analysis — BAR vs Athletic

Frame-level ball possession for **FC Barcelona (home, `H`) vs Athletic Club (away, `A`)**, LaLiga 2024-25, matchday 2.

A submission is one label per tactical-video frame (`199,319` rows at 25 fps):

| label | meaning |
|---|---|
| `H` | Barcelona in possession, ball alive |
| `A` | Athletic Club in possession, ball alive |
| `D` | dead ball |
| `X` | ungraded frame, or a deliberate abstention |

This notebook maps the files, the frame timeline, and the only labelled file that ships in the repo: the **public ground truth** for the first 5 minutes of play (frames `11185–18684`). Tracking, homography and Smart Tagging are gated. Those sections run when the bundle is unpacked into `data/` (or `MTAH_DATA_DIR`).


## Setup

Paths come from `games.py`, the same module the loaders and the scorer use. Run this notebook from `eda/` or from the repository root.


In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "games.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import games
from games import GAME, GAMES, SOURCES, asset_paths

def resolve_data_dir():
    """The bundle is sometimes unpacked as data/tactical and sometimes as data/data/tactical."""
    candidates = []
    if os.environ.get("MTAH_DATA_DIR"):
        candidates.append(Path(os.environ["MTAH_DATA_DIR"]))
    candidates.extend([ROOT / "data", ROOT / "data" / "data"])
    for candidate in candidates:
        if (candidate / "tactical").is_dir():
            return candidate
    return ROOT / "data"

DATA_DIR = resolve_data_dir()
games.DATA_DIR = str(DATA_DIR)

FPS = GAMES[GAME]["fps"]
N_FRAMES = GAMES[GAME]["n_video_frames"]
PITCH_M = GAMES[GAME]["pitch"]
LABELS = ["H", "A", "D", "X"]
COLORS = {"H": "#004d98", "A": "#ee1c25", "D": "#7a7a7a", "X": "#e8e8e8"}
LABEL_NAME = {
    "H": "Barcelona (home), ball alive",
    "A": "Athletic Club (away), ball alive",
    "D": "dead ball",
    "X": "ungraded / abstain",
}

pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)
plt.rcParams.update({
    "figure.figsize": (11, 3.4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

print(f"{GAME}: {N_FRAMES:,} tactical frames at {FPS} fps, pitch {PITCH_M[0]} x {PITCH_M[1]} m")
print(f"repository: {ROOT}")
print(f"data directory: {DATA_DIR}")


## What is on disk

Two recordings, each with its own timeline. The **tactical** video is the index of a submission: row `i` is frame `i`. The broadcast recording is optional and is not aligned to it.

Tracking coordinates are normalised to `0–1` (multiply by the pitch in metres). ATD `frame_id` counts from 1, so `video_frame = frame_id - 1`. Smart Tagging times are seconds from the first tactical frame: `video_frame = round(seconds * 25)`. Homography `frame` already matches a submission row.


In [ ]:
catalog = {
    "public ground truth": ROOT / "evaluation/gt/BAR-ATH_possession_gt_public.csv",
    "public ground truth meta": ROOT / "evaluation/gt/BAR-ATH_possession_gt_public_meta.json",
}
for source in ("tactical", "broadcast"):
    for name, path in asset_paths(source=source).items():
        catalog[f"{source} / {name}"] = Path(path)

inventory = pd.DataFrame([
    {
        "asset": name,
        "present": path.is_file(),
        "size_mb": round(path.stat().st_size / 1e6, 2) if path.is_file() else 0.0,
        "path": str(path.relative_to(ROOT)) if str(path).startswith(str(ROOT)) else str(path),
    }
    for name, path in catalog.items()
])
inventory


In [ ]:
source_table = pd.DataFrame([
    {
        "source": name,
        "role": "submission index" if name == "tactical" else "optional, not aligned",
        "frames": spec["n_video_frames"],
        "minutes": spec["n_video_frames"] / FPS / 60,
        "assets": ", ".join(spec["assets"]),
    }
    for name, spec in SOURCES.items()
])
source_table


## Match timeline

Only the two halves are graded (`158,857` frames). Labels written before kickoff, at halftime, or after the final whistle are ignored. The public file fills those stretches with `X`.


In [ ]:
def video_spans():
    spans = []
    cursor = 0
    for half, first, last in games.half_video_blocks(GAME):
        if first > cursor:
            spans.append(("ungraded", cursor, first - 1))
        spans.append((f"half {half}", first, last))
        cursor = last + 1
    if cursor < N_FRAMES:
        spans.append(("ungraded", cursor, N_FRAMES - 1))
    names = ["before kickoff", "1st half", "halftime", "2nd half", "after full time"]
    rows = []
    for (kind, first, last), name in zip(spans, names):
        n = last - first + 1
        rows.append({
            "span": name,
            "first_frame": first,
            "last_frame": last,
            "frames": n,
            "minutes": round(n / FPS / 60, 2),
            "graded": kind.startswith("half"),
        })
    return pd.DataFrame(rows)

timeline = video_spans()
timeline


In [ ]:
fig, ax = plt.subplots(figsize=(11, 1.6))
for _, row in timeline.iterrows():
    color = "#004d98" if row.graded else "#d0d0d0"
    ax.barh(0, row.frames, left=row.first_frame, height=0.55, color=color, edgecolor="white")
    ax.text(row.first_frame + row.frames / 2, 0, row.span,
            ha="center", va="center", fontsize=8, color="white" if row.graded else "#333")
ax.set_xlim(0, N_FRAMES)
ax.set_yticks([])
ax.set_xlabel("tactical video frame")
ax.set_title("Graded frames are the two halves")
plt.tight_layout()
plt.show()

print(f"graded frames: {int(timeline.loc[timeline.graded, 'frames'].sum()):,}")
print(f"ungraded frames: {int(timeline.loc[~timeline.graded, 'frames'].sum()):,}")


## Public ground truth

`evaluation/gt/BAR-ATH_possession_gt_public.csv` has the full `199,319` rows. Real labels exist only inside the first **5 minutes of play** (frames `11185–18684`, `7,500` frames, `17` possessions). Everywhere else is `X` and is never scored.

This window checks that a pipeline lines up. It is too short to rank or tune a model: the published baselines reorder between this window and the full match.


In [ ]:
gt_path = ROOT / "evaluation/gt/BAR-ATH_possession_gt_public.csv"
meta_path = ROOT / "evaluation/gt/BAR-ATH_possession_gt_public_meta.json"
gt = pd.read_csv(gt_path)
meta = json.loads(meta_path.read_text())

assert list(gt.columns) == ["frame", "label"]
assert len(gt) == N_FRAMES
assert np.array_equal(gt["frame"].to_numpy(), np.arange(N_FRAMES))
assert set(gt["label"]) <= set(LABELS)

window_first, window_last = meta["public_window"]["first_video_frame"], meta["public_window"]["last_video_frame"]
window = gt.loc[gt["frame"].between(window_first, window_last)].copy()
print(f"{len(gt):,} rows, labels {sorted(gt['label'].unique())}")
print(f"public window: frames {window_first}–{window_last} ({len(window):,} frames, {len(window)/FPS/60:.1f} min)")
print(f"meta: {meta['n_possessions']} possessions, mean duration {meta['mean_possession_duration_s']} s")


In [ ]:
def label_summary(frame, scope):
    counts = frame["label"].value_counts().reindex(LABELS, fill_value=0)
    out = counts.rename("frames").to_frame()
    out["share"] = out["frames"] / out["frames"].sum()
    out["seconds"] = out["frames"] / FPS
    out["scope"] = scope
    out.index.name = "label"
    return out.reset_index()

summary = pd.concat([
    label_summary(gt, "full file"),
    label_summary(window, "public 5 min"),
], ignore_index=True)
summary


In [ ]:
alive = window[window["label"].isin(["H", "A"])]
share_h = (window["label"] == "H").sum() / len(alive)
share_d = (window["label"] == "D").mean()
print(f"of alive frames, Barcelona has {share_h:.1%} and Athletic {1 - share_h:.1%}")
print(f"dead ball is {share_d:.1%} of the public window ({(window['label']=='D').sum()/FPS:.1f} s)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
order = ["H", "A", "D"]
counts = window["label"].value_counts().reindex(order)
axes[0].bar(order, counts / FPS, color=[COLORS[k] for k in order])
axes[0].set_ylabel("seconds")
axes[0].set_title("Time by label, public window")
alive_counts = window["label"].value_counts().reindex(["H", "A"])
axes[1].bar(["H", "A"], alive_counts / alive_counts.sum(), color=[COLORS["H"], COLORS["A"]])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("share of alive frames")
axes[1].set_title("Possession share, ball alive")
plt.tight_layout()
plt.show()


## Spells

A spell is a maximal run of one label. Possessions are the `H` and `A` spells. `D` spells are stoppages, not possessions. The ranking metric matches possessions by temporal overlap, so the count and the length of these runs matter more than the frame totals above.


In [ ]:
def spells_from(frame):
    labels = frame["label"].to_numpy()
    frames = frame["frame"].to_numpy()
    breaks = np.flatnonzero(labels[1:] != labels[:-1]) + 1
    bounds = np.concatenate(([0], breaks, [len(labels)]))
    rows = []
    for i in range(len(bounds) - 1):
        a, b = int(bounds[i]), int(bounds[i + 1])
        n = b - a
        rows.append({
            "start_frame": int(frames[a]),
            "end_frame": int(frames[b - 1]),
            "label": labels[a],
            "frames": n,
            "seconds": n / FPS,
        })
    spells = pd.DataFrame(rows)
    origin = int(spells["start_frame"].iloc[0])
    spells["start_s"] = (spells["start_frame"] - origin) / FPS
    spells["video_clock"] = spells["start_frame"].map(
        lambda f: f"{int(f / FPS // 60):02d}:{f / FPS % 60:05.2f}"
    )
    return spells

spells = spells_from(window)
possessions = spells[spells["label"].isin(["H", "A"])].copy()
stoppages = spells[spells["label"] == "D"].copy()
print(f"{len(spells)} spells: {len(possessions)} possessions, {len(stoppages)} dead-ball runs")
spells


In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.2))
for _, row in spells.iterrows():
    ax.barh(0, row.seconds, left=row.start_s, height=0.62, color=COLORS[row.label], edgecolor="white", linewidth=0.4)
handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[k]) for k in ("H", "A", "D")]
ax.legend(handles, [LABEL_NAME[k] for k in ("H", "A", "D")], loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.45))
ax.set_yticks([])
ax.set_xlim(0, len(window) / FPS)
ax.set_xlabel("seconds from the first graded frame")
ax.set_title("Public window, one bar per spell")
plt.tight_layout()
plt.show()


In [ ]:
duration = (
    possessions.groupby("label")["seconds"]
    .agg(n="count", total_s="sum", mean_s="mean", median_s="median", min_s="min", max_s="max")
    .reindex(["H", "A"])
)
display(duration)
print(
    f"all possessions: n={len(possessions)}, mean={possessions['seconds'].mean():.2f}s, "
    f"median={possessions['seconds'].median():.2f}s"
)
print(
    f"dead-ball runs: n={len(stoppages)}, mean={stoppages['seconds'].mean():.2f}s, "
    f"total={stoppages['seconds'].sum():.1f}s"
)

fig, ax = plt.subplots(figsize=(8, 3.2))
bins = np.linspace(0, possessions["seconds"].max() + 1, 12)
for label in ("H", "A"):
    ax.hist(possessions.loc[possessions.label == label, "seconds"], bins=bins,
            alpha=0.75, label=label, color=COLORS[label])
ax.axvline(possessions["seconds"].median(), color="black", ls="--", lw=1, label="median")
ax.set_xlabel("possession length (s)")
ax.set_ylabel("count")
ax.set_title("Possession lengths in the public window")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


The mean (~14 s) sits well above the median (~7.6 s). A few long Barcelona spells (about 29–42 s) pull the mean up, and three possessions last under 4 seconds. A frame-level metric barely notices a boundary that is a few frames late; a spell metric treats a short possession that is split or merged as a miss.


## Transitions

Each change of label inside the window. `frame` is the first frame of the new label.


In [ ]:
transitions = pd.DataFrame({
    "frame": spells["start_frame"].iloc[1:].to_numpy(),
    "from": spells["label"].iloc[:-1].to_numpy(),
    "to": spells["label"].iloc[1:].to_numpy(),
    "at_s": spells["start_s"].iloc[1:].to_numpy(),
})
pair_counts = (
    transitions.groupby(["from", "to"]).size().rename("n").reset_index()
    .sort_values("n", ascending=False)
)
display(pair_counts)
transitions


In [ ]:
matrix = pd.crosstab(transitions["from"], transitions["to"]).reindex(index=["H", "A", "D"], columns=["H", "A", "D"], fill_value=0)
display(matrix)

fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(matrix.to_numpy(), cmap="Blues")
ax.set_xticks(range(3), ["H", "A", "D"])
ax.set_yticks(range(3), ["H", "A", "D"])
ax.set_xlabel("to")
ax.set_ylabel("from")
ax.set_title("Transitions in the public window")
for i in range(3):
    for j in range(3):
        ax.text(j, i, int(matrix.to_numpy()[i, j]), ha="center", va="center")
ax.grid(False)
plt.tight_layout()
plt.show()


Alive possessions change hands directly (`H→A`, `A→H`) or through a stoppage (`→D`). In this window the four dead-ball runs are all entered from `A` and left toward `H` or the end of the window. Direct turnovers are the majority of the 20 boundaries.


## Smart Tagging

Automatic phases and set pieces on the tactical video (`638` instances in the documentation). Intervals are phases, not possessions, and `Team` is `N/A` on some restarts, including every throw-in. Tags are evidence, not labels.


In [ ]:
tag_path = Path(asset_paths()["smart_tagging"])
if not tag_path.is_file():
    print(f"Smart Tagging is not unpacked.\nExpected: {tag_path}")
else:
    sys.path.insert(0, str(ROOT / "starter"))
    import load_smart_tagging

    raw_events = load_smart_tagging.load(data_dir=str(DATA_DIR))
    time_base = load_smart_tagging.check_time_base(raw_events)
    events = pd.DataFrame(raw_events)
    tags = pd.json_normalize(events["tags"])
    events = pd.concat([events.drop(columns="tags"), tags], axis=1)
    events["seconds"] = events["end_s"] - events["start_s"]
    print(f"{len(events)} instances; kickoff tags sit on the video clock: {time_base['agrees']}")

    code_counts = events["code"].value_counts()
    display(code_counts.rename("n").to_frame())
    display(events["Team"].value_counts(dropna=False).rename("n").to_frame())
    if "Type" in events.columns:
        set_pieces = events.loc[events["code"].str.upper() == "SET PIECES", "Type"]
        display(set_pieces.value_counts(dropna=False).rename("set_piece_type").to_frame())

    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.barh(code_counts.index[::-1], code_counts.values[::-1], color="#004d98")
    ax.set_xlabel("instances")
    ax.set_title("Smart Tagging codes")
    plt.tight_layout()
    plt.show()

    in_window = events[(events.end_frame >= window_first) & (events.start_frame <= window_last + 1)].copy()
    print(f"{len(in_window)} phases overlap the public 5-minute window")
    cols = [c for c in ["code", "Team", "Half", "Type", "start_frame", "end_frame", "seconds"] if c in in_window.columns]
    display(in_window[cols].reset_index(drop=True))


## Homography

One 3×3 matrix per video frame, mapping normalised pitch coordinates to pixels. `valid == 0` means the matrix is missing. Tactical homography is documented as valid on every frame; broadcast is valid on about 65.8%.


In [ ]:
def period_of(frames):
    period = np.full(len(frames), "ungraded", dtype=object)
    for half, first, last in games.half_video_blocks(GAME):
        period[(frames >= first) & (frames <= last)] = f"half {half}"
    return period

def summarise_homography(source):
    path = Path(asset_paths(source=source)["homography"])
    if not path.is_file():
        print(f"{source}: homography not unpacked ({path})")
        return None
    table = pd.read_csv(path, usecols=["frame", "reprojection_error", "valid"])
    table["period"] = period_of(table["frame"].to_numpy()) if source == "tactical" else "broadcast timeline"
    grouped = (
        table.groupby("period")
        .apply(lambda g: pd.Series({
            "frames": len(g),
            "valid_share": g["valid"].mean(),
            "error_median": g.loc[g["valid"] == 1, "reprojection_error"].median(),
            "error_p99": g.loc[g["valid"] == 1, "reprojection_error"].quantile(0.99),
        }), include_groups=False)
        .reset_index()
    )
    print(f"{source}: {len(table):,} rows, valid {table['valid'].mean():.1%}")
    return grouped

for source in ("tactical", "broadcast"):
    summary_h = summarise_homography(source)
    if summary_h is not None:
        display(summary_h)


## Tracking (ATD)

The raw file is one line per video frame: `frame_id:player;player;...:ball`, each group `x,y,speed`, with `NaN` when that object is missing. Coordinates are normalised `0–1`. There are dozens of track slots for 22 players on the pitch. A slot is a tracklet (`Track 1` …), and its shirt number is the slot index, not a person.

The ball is missing on a large share of frames. The pass below streams the file once and records visibility, how many tracks are alive, and a sample of ball positions. The match videos themselves are not in this bundle.


In [ ]:
import xml.etree.ElementTree as ET

def track_table(source):
    meta_path = Path(asset_paths(source=source)["atd_meta"])
    root = ET.parse(meta_path).getroot()
    teams = [(team.get("id"), team.findtext("Name")) for team in root.iter("Team")]
    home_id = teams[0][0]
    rows = []
    for player in root.iter("Player"):
        rows.append({
            "slot": int(player.get("id")),
            "name": player.findtext("Name"),
            "shirt": player.findtext("ShirtNumber"),
            "team": "H" if player.get("teamId") == home_id else "A",
            "club": teams[0][1] if player.get("teamId") == home_id else teams[1][1],
        })
    table = pd.DataFrame(rows)
    print(f"{source}: {teams[0][1]} (H) vs {teams[1][1]} (A), {len(table)} track slots")
    print(table["team"].value_counts().to_dict())
    return table

def stream_atd(source="tactical", stride=25):
    """One pass over the raw ATD. Returns per-frame arrays plus a stride sample of the ball."""
    path = Path(asset_paths(source=source)["atd_raw"])
    if not path.is_file():
        print(f"{source}: ATD not unpacked ({path})")
        return None
    slots = track_table(source)
    home_slot = (slots["team"] == "H").to_numpy()

    n_home, n_away, ball_on = [], [], []
    sample_frame, sample_ball_x, sample_ball_y, sample_players = [], [], [], []
    with path.open() as handle:
        for n, line in enumerate(handle):
            frame_id, players, ball = line.rstrip("\n").split(":")
            live_h = live_a = 0
            for slot, group in enumerate(players.split(";")):
                if group and group[0] != "N":
                    if home_slot[slot]:
                        live_h += 1
                    else:
                        live_a += 1
            has_ball = ball[0] != "N"
            n_home.append(live_h)
            n_away.append(live_a)
            ball_on.append(has_ball)
            if n % stride == 0 and has_ball:
                x, y, _speed = ball.split(",")
                sample_frame.append(int(frame_id) - 1)
                sample_ball_x.append(float(x))
                sample_ball_y.append(float(y))
                sample_players.append(live_h + live_a)

    home = np.asarray(n_home, dtype=np.int16)
    away = np.asarray(n_away, dtype=np.int16)
    ball = np.asarray(ball_on)
    players = home + away
    n = len(ball)
    graded = np.zeros(n, dtype=bool)
    if source == "tactical":
        for _half, first, last in games.half_video_blocks(GAME):
            graded[first:last + 1] = True
    else:
        graded[:] = True

    missing = ~ball
    # gap lengths on the frames where a submission is graded
    flag = (missing & graded).astype(np.int8)
    edges = np.diff(np.concatenate(([0], flag, [0])))
    gap_lengths = np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)

    print(f"{source}: {n:,} records")
    print(f"  ball visible: {ball.mean():.1%} of all frames, {ball[graded].mean():.1%} of graded frames")
    print(f"  players on graded frames: mean {players[graded].mean():.2f}, median {np.median(players[graded]):.0f}")
    print(f"  home/away tracks on graded frames: {home[graded].mean():.1f} / {away[graded].mean():.1f}")
    if len(gap_lengths):
        print(
            f"  ball gaps on graded frames: {len(gap_lengths):,} runs, "
            f"median {np.median(gap_lengths)/FPS:.2f}s, max {gap_lengths.max()/FPS:.1f}s"
        )
    sample = pd.DataFrame({
        "video_frame": sample_frame,
        "ball_x": sample_ball_x,
        "ball_y": sample_ball_y,
        "n_players": sample_players,
    })
    return {
        "sample": sample,
        "players": players,
        "home": home,
        "away": away,
        "ball": ball,
        "graded": graded,
        "gap_lengths": gap_lengths,
    }

tactical = stream_atd("tactical")


In [ ]:
if tactical is not None:
    players = tactical["players"]
    graded = tactical["graded"]
    sample = tactical["sample"]
    graded_sample = sample[sample["video_frame"].map(lambda f: bool(graded[f]))]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    step = 25
    axes[0].plot(np.arange(0, len(players), step) / FPS / 60, players[::step], color="#333", lw=0.7)
    for _half, first, last in games.half_video_blocks(GAME):
        axes[0].axvspan(first / FPS / 60, last / FPS / 60, color="#004d98", alpha=0.08)
    axes[0].set_xlabel("minutes from the start of the tactical video")
    axes[0].set_ylabel("live tracks")
    axes[0].set_title("How many objects the tracker is holding")

    axes[1].scatter(graded_sample["ball_x"], 1 - graded_sample["ball_y"], s=4, alpha=0.25, c="#004d98", linewidths=0)
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].set_aspect("equal")
    axes[1].set_xlabel("pitch x (0–1)")
    axes[1].set_ylabel("pitch y (0–1)")
    axes[1].set_title("Ball positions during the two halves")
    axes[1].grid(False)
    plt.tight_layout()
    plt.show()

    gaps_s = tactical["gap_lengths"] / FPS
    if len(gaps_s):
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.hist(gaps_s, bins=40, color="#7a7a7a")
        ax.set_xlabel("ball-missing gap during play (seconds)")
        ax.set_ylabel("gaps")
        ax.set_title("How long the tracker loses the ball")
        plt.tight_layout()
        plt.show()


In [ ]:
broadcast = stream_atd("broadcast")


## What the files support

The public labels cover five minutes and 17 possessions. The tracking covers the whole match, but the ball is absent on a large fraction of frames and the track slots are not players. Smart Tagging adds a team and a phase on each interval, including stretches the public file marks dead, and it is automatic rather than ground truth. Homography is the link from a pitch coordinate to a pixel; the tactical file is usable throughout, the broadcast file is not.

The two videos are not in `data/`. Everything above is the tracking, the tags, the homographies, and the public labels.
